# Mini Workshop — ONNX

> *PyTorch is the language you wrote your model in. ONNX is the language your model travels in.*

In this Mini, we'll:
1. **Run** SceneSeg (an Autoware perception model) **as PyTorch**
2. **Convert** it to ONNX yourself with `torch.onnx.export()`
3. **Run** your exported ONNX with ONNX Runtime, and verify the math matches PyTorch

The big Workshop applies the same export pattern, then takes the ONNX further into TensorRT FP16 and INT8.


## Setup


In [ ]:
!pip install onnxruntime-gpu gdown onnxscript onnx -q


In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import onnxruntime as ort
import glob, os

print(f"PyTorch {torch.__version__} | ORT {ort.__version__}")


## Download — SceneSeg model & a Waymo frame

We use the same SceneSeg model the big Workshop uses, so everything connects. Just two files:
- `SceneSeg_traced.pt` — the PyTorch (TorchScript) model
- A few Waymo driving frames to run inference on


In [ ]:
!mkdir -p /content/data /content/models

# SceneSeg PyTorch (traced)
!gdown '1G2pKrjEGLGY1ouQdNPh11N-5LlmDI7ES' -O /content/models/SceneSeg_traced.pt

# Waymo driving frames
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip -O /content/data/waymo.zip
!unzip -qq /content/data/waymo.zip -d /content/data/

print(f"\nModel: {os.path.getsize('/content/models/SceneSeg_traced.pt')/1e6:.1f} MB")


In [ ]:
# SceneSeg expects 320 x 640 input. (You can confirm this by opening
# the .pt in Netron, or by inspecting the ONNX after we export it below.)
H, W = 320, 640


## Load a Waymo frame

We'll run the same image through both runtimes (PyTorch then ONNX) and compare.


In [ ]:
frames = sorted(glob.glob('/content/data/night/front_images_night/*.jpg'))
frame_path = frames[100]

frame_bgr = cv2.imread(frame_path)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

preprocess = T.Compose([
    T.Resize((H, W)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

x    = preprocess(Image.fromarray(frame_rgb)).unsqueeze(0)  # PyTorch tensor
x_np = x.numpy()                                            # numpy for ONNX Runtime

plt.imshow(cv2.resize(frame_rgb, (W, H))); plt.axis('off')
plt.title('Input: Waymo driving frame'); plt.show()


In [ ]:
COLORS = np.array([
    [240,  40,  40],   # 0 background
    [180,  60, 200],   # 1 foreground (cars / pedestrians)
    [ 80, 200,  80],   # 2 drivable road
], dtype=np.uint8)


---
## Part 1 — Run SceneSeg as PyTorch

`torch.jit.load` reads the TorchScript file directly — no class definition needed.


In [ ]:
model = torch.jit.load('/content/models/SceneSeg_traced.pt', map_location='cpu')
model.eval()

with torch.no_grad():
    pt_out = model(x).numpy()

pt_map = np.argmax(pt_out[0], axis=0)
print(f"PyTorch output shape: {pt_out.shape}")

# Visual: input | segmentation | overlay
input_img = cv2.resize(frame_rgb, (W, H))
overlay   = cv2.addWeighted(input_img, 0.5, COLORS[pt_map], 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(input_img);     axes[0].set_title('Input frame');           axes[0].axis('off')
axes[1].imshow(COLORS[pt_map]); axes[1].set_title('PyTorch — segmentation'); axes[1].axis('off')
axes[2].imshow(overlay);       axes[2].set_title('PyTorch — overlay');     axes[2].axis('off')
plt.tight_layout(); plt.show()


---
## Part 2 — Export PyTorch → ONNX

Autoware pre-exports SceneSeg to ONNX so any team in the world can ship it on any runtime. Now we'll do the same export ourselves.

`torch.onnx.export()` needs four things:
1. **The model** — must be in `eval()` mode
2. **A dummy input** — a tensor with the shape the model expects (it traces the graph by running this through)
3. **An output filename**
4. **An opset version** — which set of ONNX operators to use. Opset 17 is a safe modern choice for TensorRT.

Two extras worth knowing:
- `input_names` / `output_names` — readable names for the graph (much nicer than `input.1`).
- `dynamo=False` — use the legacy exporter. The new dynamo-based default doesn't yet support TorchScript modules.


In [ ]:
# TODO 1 — Build a dummy input with the shape the model expects.
# Hint: batch=1, 3 channels, H × W (defined above).
dummy = ...

# TODO 2 — Call torch.onnx.export with:
#   - model and dummy from above
#   - output path '/content/models/SceneSeg_MINE.onnx'
#   - opset_version=17
#   - input_names=['image'], output_names=['segmentation']
#   - dynamo=False  (legacy exporter — required because our model is TorchScript)
torch.onnx.export(...)

print(f"Exported: {os.path.getsize('/content/models/SceneSeg_MINE.onnx') / 1e6:.1f} MB")


---
## Part 3 — Run YOUR ONNX

Same input, same expected output — but the model now runs through ONNX Runtime instead of PyTorch.

The whole point of ONNX export is that the math doesn't change: same numbers in, same numbers out (down to floating-point noise from kernel reordering).


In [ ]:
# TODO 3 — Load your exported ONNX with onnxruntime.
sess = ...

# TODO 4 — Run inference on x_np (input name is 'image').
ort_out = ...
ort_map = np.argmax(ort_out[0], axis=0)

# TODO 5 — Visualize: input | PyTorch overlay | your ONNX overlay (3 panels).
# Hint: cv2.addWeighted(input_img, 0.5, COLORS[some_map], 0.5, 0).

# TODO 6 — Compare to PyTorch output (pt_out from Part 1).
# Print max + mean abs diff, and whether np.allclose passes (atol=1e-4).


---
## Bonus — What's actually inside an ONNX file?

ONNX files are graphs — nodes (operators) connected by tensors. Let's open the file we just exported and see what's in there.


In [ ]:
import onnx
m = onnx.load('/content/models/SceneSeg_MINE.onnx')

# Count nodes and unique operator types
op_counts = {}
for node in m.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

total_ops    = sum(op_counts.values())
unique_ops   = len(op_counts)
top_5        = sorted(op_counts.items(), key=lambda kv: -kv[1])[:5]

print(f"Total nodes:       {total_ops}")
print(f"Unique op types:   {unique_ops}")
print(f"Inputs / outputs:  {len(m.graph.input)} / {len(m.graph.output)}")
print(f"\nTop 5 ops by count:")
for op, n in top_5:
    print(f"  {op:<20} {n}")

print(f"\nTip: open SceneSeg_MINE.onnx in Netron (https://netron.app) for a visual graph.")


---
## What's next

You ran SceneSeg as PyTorch, exported it to ONNX yourself, ran your exported version, and verified the math survived.

In the Workshop (`Model_Deployment.ipynb`), we take this same export and:
1. Compile it into a **TensorRT engine** for the GPU
2. Calibrate it to **INT8** using real driving frames
3. Benchmark all 5 runtimes side by side and produce a comparison video

That's the full deployment story — see you there.
